# 피처 엔지니어링 v2

## 구성
| 단계 | 내용 |
|------|------|
| 1 | 데이터 로드 & 기본 시간 피처 |
| 2 | **기존** 피처 (lag1~3, diff, ratio, rolling_mean_3, sin/cos_month, normal_ratio) |
| 3 | **추가** lag 피처 (lag6, lag12, lag36, yoy_change) |
| 4 | **추가** 롤링·EWM 피처 |
| 5 | **추가** 변동성 피처 |
| 6 | **추가** 시간·이벤트 피처 |
| 7 | **추가** 평년 대비 피처 |
| 8 | 최종 피처 목록 정리 & 저장 |

---
**그룹 키**: `['품목', '품목명', '거래처명', '단위']`  
**타깃**: `실제가격(원)` (log 변환 여부는 USE_LOG 플래그로 제어)

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ===============================
# 설정
# ===============================
TRAIN_PATH = 'train_target_only_cleaned (1).csv'
SAVE_PATH  = 'train_features_v2.csv'
USE_LOG    = False   # True로 바꾸면 log1p 변환 타깃 사용

GROUP_COLS = ['품목', '품목명', '거래처명', '단위']
CAT_COLS   = ['품목명', '거래처명', '단위']
TARGET_COL = '실제가격(원)'

---
## 1. 데이터 로드 & 기본 시간 피처

In [2]:
df = pd.read_csv(TRAIN_PATH)
print('원본 shape:', df.shape)
df.head(3)

원본 shape: (1425, 7)


,시점,품목명,품종명,거래단위,등급,평년 평균가격(원),평균가격(원)
0,201801상순,건고추,화건,30 kg,상품,381666.666667,590000.0
1,201801중순,건고추,화건,30 kg,상품,380809.666667,590000.0
2,201801하순,건고추,화건,30 kg,상품,380000.000000,590000.0


In [4]:
# ── 시간 파싱 ──────────────────────────────────────────────
순_map = {'상순': 0, '중순': 1, '하순': 2}

df['year']    = df['연도'].str[:4].astype(int)
df['month']   = df['연도'].str[4:6].astype(int)
df['순_str']  = df['연도'].str[6:]
df['순_num']  = df['순_str'].map(순_map)            # 0/1/2

# 전체 정렬용 시간 인덱스 (1순 = 1단위)
df['time_idx'] = df['year'] * 36 + (df['month'] - 1) * 3 + df['순_num']

# 그룹 정렬
df = df.sort_values(GROUP_COLS + ['time_idx']).reset_index(drop=True)

# ── 타깃 ──────────────────────────────────────────────────
df['y'] = np.log1p(df[TARGET_COL]) if USE_LOG else df[TARGET_COL]

print('time_idx 범위:', df['time_idx'].min(), '~', df['time_idx'].max())
print('연도:', sorted(df['year'].unique()))

KeyError: '연도'

---
## 2. 기존 피처

| 피처 | 설명 |
|------|------|
| `lag1~3` | 직전 1~3순 가격 |
| `diff1, diff2` | lag 연속 차분 (변화량) |
| `ratio1, ratio2` | lag 비율 (상대 변화율) |
| `rolling_mean_3` | 3순 롤링 평균 |
| `sin_month, cos_month` | 월 순환 인코딩 |
| `normal_ratio` | (실제 − 평년) / 평년 |

In [ ]:
g = df.groupby(GROUP_COLS)['y']

# ── Lag 1~3 ───────────────────────────────────────────────
for lag in [1, 2, 3]:
    df[f'lag{lag}'] = g.shift(lag)

# ── 차분 (절대 변화량) ────────────────────────────────────
df['diff1'] = df['lag1'] - df['lag2']
df['diff2'] = df['lag2'] - df['lag3']

# ── 비율 (상대 변화율) ────────────────────────────────────
df['ratio1'] = df['lag1'] / (df['lag2'] + 1e-6)
df['ratio2'] = df['lag2'] / (df['lag3'] + 1e-6)

# ── 3순 롤링 평균 ─────────────────────────────────────────
df['rolling_mean_3'] = g.transform(lambda x: x.shift(1).rolling(3).mean())

# ── 월 순환 인코딩 ────────────────────────────────────────
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

# ── 평년 대비 비율 ────────────────────────────────────────
df['normal_ratio'] = (
    (df[TARGET_COL] - df['평년가격(원)']) / (df['평년가격(원)'] + 1e-6)
)

print('기존 피처 생성 완료')

---
## 3. 추가 — 중기·전년 Lag

| 피처 | 설명 | 기존과의 차이 |
|------|------|---------------|
| `lag6` | 2개월(6순) 전 가격 | 중기 사이클 포착 |
| `lag12` | 4개월(12순) 전 가격 | 분기 효과 포착 |
| `lag36` | 전년 동기 가격 | 연간 계절성 직접 포착 |
| `yoy_change` | (현재 − lag36) / lag36 | 전년 동기 대비 변화율 |

In [ ]:
for lag in [6, 12, 36]:
    df[f'lag{lag}'] = g.shift(lag)

# 전년 동기 대비 변화율
df['yoy_change'] = (df['lag1'] - df['lag36']) / (df['lag36'] + 1e-6)

print('lag6 결측:', df['lag6'].isna().sum())
print('lag12 결측:', df['lag12'].isna().sum())
print('lag36 결측:', df['lag36'].isna().sum())

---
## 4. 추가 — 롤링 · EWM 피처

| 피처 | 설명 | 기존과의 차이 |
|------|------|---------------|
| `rolling_mean_6` | 6순 롤링 평균 | rolling_mean_3보다 더 평활화 |
| `rolling_std_3` | 3순 롤링 표준편차 | 국소 변동성 포착 |
| `rolling_std_6` | 6순 롤링 표준편차 | 중기 변동성 포착 |
| `ewm_span6` | 지수가중이동평균 (span=6) | 최근 값에 더 높은 가중치 |

In [ ]:
# 롤링 (shift(1)로 leakage 방지)
df['rolling_mean_6'] = g.transform(lambda x: x.shift(1).rolling(6).mean())
df['rolling_std_3']  = g.transform(lambda x: x.shift(1).rolling(3).std())
df['rolling_std_6']  = g.transform(lambda x: x.shift(1).rolling(6).std())

# EWM (지수가중이동평균)
df['ewm_span6'] = g.transform(lambda x: x.shift(1).ewm(span=6, adjust=False).mean())

print('롤링·EWM 피처 생성 완료')

---
## 5. 추가 — 변동성 · 모멘텀 피처

| 피처 | 설명 | 기존과의 차이 |
|------|------|---------------|
| `rolling_cv6` | 6순 롤링 변동계수(CV) | 변동성의 상대적 크기 |
| `price_momentum` | lag1/lag3 − 1 | 단기 모멘텀 방향성 |
| `price_zscore` | (lag1 − 품목별 평균) / 품목별 std | 품목 간 스케일 차이 제거 |

In [ ]:
# 롤링 변동계수 (std / mean)
rolling_mean = g.transform(lambda x: x.shift(1).rolling(6).mean())
rolling_std  = g.transform(lambda x: x.shift(1).rolling(6).std())
df['rolling_cv6'] = rolling_std / (rolling_mean + 1e-6)

# 단기 모멘텀
df['price_momentum'] = df['lag1'] / (df['lag3'] + 1e-6) - 1

# 품목별 z-score (학습 데이터 내 통계 기준)
item_mean = df.groupby('품목')['y'].transform('mean')
item_std  = df.groupby('품목')['y'].transform('std')
df['price_zscore'] = (df['lag1'] - item_mean) / (item_std + 1e-6)

print('변동성·모멘텀 피처 생성 완료')

---
## 6. 추가 — 시간 · 이벤트 피처

| 피처 | 설명 | 기존과의 차이 |
|------|------|---------------|
| `sun_sin`, `sun_cos` | 순(1~3) 순환 인코딩 | 순_num(정수)의 순환 구조 개선 |
| `year_trend` | 연도 − 2018 | 선형 트렌드 직접 인코딩 |
| `is_2020` | 2020년 더미 | 코로나 충격 명시적 분리 |

In [ ]:
# 순 순환 인코딩 (1~3 → 0/1/2로 매핑 후 주기=3)
df['sun_sin'] = np.sin(2 * np.pi * df['순_num'] / 3)
df['sun_cos'] = np.cos(2 * np.pi * df['순_num'] / 3)

# 연도 트렌드
df['year_trend'] = df['year'] - 2018

# 코로나 더미
df['is_2020'] = (df['year'] == 2020).astype(int)

print('시간·이벤트 피처 생성 완료')

---
## 7. 추가 — 평년 대비 피처

| 피처 | 설명 | 기존과의 차이 |
|------|------|---------------|
| `normal_ratio_lag1` | 1순 전 normal_ratio | 편차의 지속성(관성) 포착 |
| `normal_ratio_roll3` | 3순 롤링 평균 normal_ratio | 단기 편차 추세 요약 |
| `is_above_normal` | normal_ratio > 0 더미 | 이진 방향성 피처 |
| `log_normal_price` | log1p(평년가격) | 평년가 스케일 안정화 |

In [ ]:
g_nr = df.groupby(GROUP_COLS)['normal_ratio']

df['normal_ratio_lag1']  = g_nr.shift(1)
df['normal_ratio_roll3'] = g_nr.transform(lambda x: x.shift(1).rolling(3).mean())
df['is_above_normal']    = (df['normal_ratio'] > 0).astype(int)
df['log_normal_price']   = np.log1p(df['평년가격(원)'])

print('평년 대비 피처 생성 완료')

---
## 8. 최종 피처 목록 정리 & 저장

In [ ]:
# ── 피처 목록 ─────────────────────────────────────────────
FEATURE_COLS = [
    # 카테고리
    '품목명', '거래처명', '단위',

    # [기존] lag
    'lag1', 'lag2', 'lag3',

    # [기존] 차분·비율
    'diff1', 'diff2',
    'ratio1', 'ratio2',

    # [기존] 롤링
    'rolling_mean_3',

    # [기존] 시간
    'month', '순_num', 'sin_month', 'cos_month', 'time_idx',

    # [기존] 평년
    'normal_ratio', '평년가격(원)',

    # [추가] 중기·전년 lag
    'lag6', 'lag12', 'lag36',
    'yoy_change',

    # [추가] 롤링·EWM
    'rolling_mean_6',
    'rolling_std_3', 'rolling_std_6',
    'ewm_span6',

    # [추가] 변동성·모멘텀
    'rolling_cv6',
    'price_momentum',
    'price_zscore',

    # [추가] 시간·이벤트
    'sun_sin', 'sun_cos',
    'year_trend', 'is_2020',

    # [추가] 평년 대비
    'normal_ratio_lag1', 'normal_ratio_roll3',
    'is_above_normal',
    'log_normal_price',
]

TARGET_COL_FINAL = 'y'

# ── 결측 제거 (lag36 기준) ─────────────────────────────────
# lag36이 없는 초기 행은 제거 (전체 1,425 중 일부 손실 예상)
df_feat = df.dropna(subset=['lag1','lag2','lag3']).copy()
df_feat_full = df.dropna(subset=['lag36']).copy()  # lag36까지 요구하는 엄격 버전

print(f'lag1~3 기준 결측 제거 후: {df_feat.shape[0]}행')
print(f'lag36 기준 결측 제거 후:  {df_feat_full.shape[0]}행')

print(f'\n피처 수: {len(FEATURE_COLS)}개')
print('\n[기존]', [c for c in FEATURE_COLS if c in ['lag1','lag2','lag3','diff1','diff2','ratio1','ratio2','rolling_mean_3','sin_month','cos_month','month','순_num','time_idx','normal_ratio','평년가격(원)','품목명','거래처명','단위']])
print('\n[추가]', [c for c in FEATURE_COLS if c not in ['lag1','lag2','lag3','diff1','diff2','ratio1','ratio2','rolling_mean_3','sin_month','cos_month','month','순_num','time_idx','normal_ratio','평년가격(원)','품목명','거래처명','단위']])

In [ ]:
# 피처 결측 현황 체크
missing = df_feat[FEATURE_COLS].isna().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# 저장 (lag1~3 기준 결측 제거 버전 — 모델링에 사용)
save_cols = GROUP_COLS + ['year', 'month', '순_str', '순_num', 'time_idx', TARGET_COL_FINAL] + FEATURE_COLS
save_cols = list(dict.fromkeys(save_cols))  # 중복 제거

df_feat[save_cols].to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')
print(f'저장 완료: {SAVE_PATH}  ({df_feat.shape[0]}행 × {len(save_cols)}열)')
df_feat[save_cols].head(3)